In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys
from pathlib import Path

sys.path.append(str(Path("../models").resolve()))

from pytorch_deep_model import RainPredictor

In [2]:
df = pd.read_parquet("../data/processed/historical_training_data_2025.parquet")

In [3]:
features = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure"
]

X = df[features]

df["rain"] = (df["rain"] > 0).astype(int)
y = df["rain"]



scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [4]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [5]:
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [6]:
model = RainPredictor()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [7]:
epochs = 100

for epoch in range(epochs):

    for X_batch, y_batch in train_loader:

        pred = model(X_batch)

        loss = criterion(pred, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: {loss.item():.4f}")

Epoch 0: 0.1807
Epoch 10: 0.2201
Epoch 20: 0.3036
Epoch 30: 0.4656
Epoch 40: 0.3754
Epoch 50: 0.1853
Epoch 60: 0.1970
Epoch 70: 0.0958
Epoch 80: 0.1523
Epoch 90: 0.2685


In [8]:
model.eval()

with torch.no_grad():

    probs = model(X_test)

    predictions = (probs > 0.5).float()

accuracy = (predictions == y_test).float().mean()

print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.897


In [13]:
torch.save(model.state_dict(), "../models/model_weights_v1/base_ff_model_v1.pth")